# TP 3 — Mini-pipeline Bronze → Silver → Gold
**Durée indicative : 50–70 min**

Objectifs : relier ingestion, qualité, jointure et indicateurs. Ici les trois couches sont des **DataFrames** ; la sauvegarde Parquet est une extension facultative.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


In [ ]:
from pathlib import Path
import csv
from pyspark.sql import functions as F

# Databricks Runtime 14+ sets the current directory to the notebook's folder.
DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}. Open the notebook from the Git folder."

def read_small_csv(filename):
    # Python reads the small Git-tracked CSV on the driver; Spark does the processing.
    # For big data in production, read directly from cloud storage instead.
    with (DATA_DIR / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return spark.createDataFrame(rows)


## Bronze : garder la donnée brute
D’où vient chaque ligne ? Que ferait-on de la date d’ingestion en production ?

In [ ]:
bronze_admissions = read_small_csv("admissions_synthetiques.csv").withColumn("date_ingestion", F.current_date())
bronze_patients = read_small_csv("patients_synthetiques.csv")
bronze_admissions.show(5)

## Silver : types et règles qualité
Quel compromis entre rejet et correction ? Réutilisez les règles du TP 2.

In [ ]:
silver_admissions = (bronze_admissions
    .withColumn("date_admission", F.to_date("date_admission"))
    .withColumn("duree_minutes", F.col("duree_minutes").cast("int"))
    .dropDuplicates(["admission_id"])
    .filter(F.col("service").isNotNull() & (F.col("service") != "") & (F.col("duree_minutes") >= 0))
    .join(bronze_patients.select("patient_id"), "patient_id", "inner"))
print("Bronze:", bronze_admissions.count(), "Silver:", silver_admissions.count())

## Gold : indicateur de pilotage
Construisez un résultat simple : nombre de passages par date et par service.

In [ ]:
gold_activite = (silver_admissions.groupBy("date_admission", "service")
    .agg(F.count("*").alias("nb_admissions"), F.round(F.avg("duree_minutes"), 1).alias("duree_moyenne_min"))
    .orderBy("date_admission", "service"))
gold_activite.show(20)

## Contrôles finaux
Un indicateur métier crédible nécessite des règles documentées et une cohérence du nombre de lignes.

In [ ]:
assert gold_activite.filter(F.col("nb_admissions") <= 0).count() == 0
assert silver_admissions.filter(F.col("duree_minutes") < 0).count() == 0
print("Contrôles de base OK")

## Extension facultative — matérialiser les couches en Parquet
⚠️ Chaque étudiant doit personnaliser `OUTPUT_ROOT`. Il n’est **pas** nécessaire de le faire pour terminer ce TP.

In [ ]:
# IMPORTANT: configure one writable Unity Catalog Volume before this step.
# Use a unique subdirectory to avoid overwriting a classmate's output.
OUTPUT_ROOT = "/Volumes/workspace/default/episen_volume/CHANGE_ME"
assert "CHANGE_ME" not in OUTPUT_ROOT, "Replace CHANGE_ME with your personal identifier."


In [ ]:
bronze_admissions.write.mode("overwrite").parquet(OUTPUT_ROOT + "/bronze_admissions")
silver_admissions.write.mode("overwrite").parquet(OUTPUT_ROOT + "/silver_admissions")
gold_activite.write.mode("overwrite").parquet(OUTPUT_ROOT + "/gold_activite")

### Restitution en 3 minutes
1. Dessiner Sources → Bronze → Silver → Gold → Tableau de bord.
2. Citer deux problèmes de qualité et une règle métier.
3. Expliquer pourquoi le pipeline doit être **orchestré**, **surveillé**, et **relançable**.